# Sélection de Variables (SHAP / Feature Importance)

Ce notebook analyse l'importance des features TF-IDF et sélectionne les plus discriminantes pour la classification de sentiment des avis Yelp.

**Objectif** : Identifier les features les plus importantes (via Random Forest + SHAP), sélectionner un sous-ensemble optimal, et comparer les performances avant/après sélection.

### Checklist SAE-118
- [ ] Calculer feature importance (Random Forest ou SHAP)
- [ ] Visualiser les top-N features
- [ ] Sélectionner un sous-ensemble de features
- [ ] Ré-entraîner un modèle avec features sélectionnées
- [ ] Comparer performance avant/après sélection
- [ ] Notebook exécutable sans erreur

## 0. Imports et Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_selection import SelectFromModel, SelectKBest, chi2

import joblib
import warnings
warnings.filterwarnings('ignore')

# Chemins
DATA_DIR   = '../../data/cleaned'
MODELS_DIR = '../../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

print('Imports OK')

## 1. Chargement et Préparation des Données

In [ ]:
print('Chargement des données...')
try:
    df = pd.read_parquet(os.path.join(DATA_DIR, 'reviews_clean.parquet'), engine='fastparquet')
except Exception:
    df = pd.read_parquet(os.path.join(DATA_DIR, 'reviews_clean.parquet'))

df = df.dropna(subset=['text', 'stars'])
print(f'Dimensions du dataset : {df.shape}')
print(f"Distribution de la cible (stars) :\n{df['stars'].value_counts(normalize=True).sort_index()}")

In [ ]:
# Échantillonnage pour la performance locale
SAMPLE_SIZE = 10_000
df_sample = df.sample(n=SAMPLE_SIZE, random_state=42)

X = df_sample['text']
y = df_sample['stars']

# Split 80 / 10 / 10
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}')

## 2. Vectorisation TF-IDF (baseline)

On utilise la même configuration que dans `01-ml-tfidf.ipynb` pour disposer d'une baseline comparable.

In [ ]:
N_FEATURES_FULL = 10_000

vectorizer_full = TfidfVectorizer(
    max_features=N_FEATURES_FULL,
    min_df=5,
    max_df=0.7,
    ngram_range=(1, 2)
)

X_train_full = vectorizer_full.fit_transform(X_train)
X_val_full   = vectorizer_full.transform(X_val)
X_test_full  = vectorizer_full.transform(X_test)

feature_names = vectorizer_full.get_feature_names_out()
print(f'Matrice TF-IDF (train) : {X_train_full.shape}')

## 3. Modèle Baseline (toutes les features)

On entraîne un modèle de référence avec **toutes** les features pour avoir un score de comparaison.

In [ ]:
clf_baseline = LogisticRegression(max_iter=500, random_state=42, n_jobs=-1)
clf_baseline.fit(X_train_full, y_train)

y_pred_baseline_val  = clf_baseline.predict(X_val_full)
y_pred_baseline_test = clf_baseline.predict(X_test_full)

acc_baseline  = accuracy_score(y_val, y_pred_baseline_val)
f1_baseline   = f1_score(y_val, y_pred_baseline_val, average='macro')

print('=== BASELINE (10 000 features) ===')
print(f'Accuracy (val) : {acc_baseline:.4f}')
print(f'F1 Macro (val) : {f1_baseline:.4f}')

## 4. Calcul de la Feature Importance (Random Forest)

On entraîne un **Random Forest** sur les features TF-IDF et on extrait l'importance de chaque feature via `feature_importances_`.

In [ ]:
print('Entraînement Random Forest pour feature importance...')
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_full, y_train)

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]  # Tri décroissant

print(f'Feature importance calculée sur {len(importances)} features.')

## 5. Visualisation des Top-N Features

In [ ]:
TOP_N = 30

top_features = [feature_names[i] for i in indices[:TOP_N]]
top_scores   = importances[indices[:TOP_N]]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(TOP_N), top_scores[::-1], color='steelblue', edgecolor='white')
ax.set_yticks(range(TOP_N))
ax.set_yticklabels(top_features[::-1], fontsize=10)
ax.set_xlabel('Importance (Random Forest)', fontsize=12)
ax.set_title(f'Top {TOP_N} Features les Plus Importantes', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Top 10 features : {top_features[:10]}')

In [ ]:
# Distribution cumulée de l'importance
sorted_importances = np.sort(importances)[::-1]
cumulative = np.cumsum(sorted_importances)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution de l'importance
axes[0].plot(sorted_importances[:500], color='steelblue')
axes[0].set_xlabel('Rang de la feature', fontsize=11)
axes[0].set_ylabel('Importance', fontsize=11)
axes[0].set_title('Distribution de l\'importance (top 500)', fontsize=12)
axes[0].grid(alpha=0.3)

# Importance cumulée
thresholds = [0.5, 0.7, 0.9, 0.95]
axes[1].plot(cumulative, color='darkorange')
for t in thresholds:
    n_feat = np.searchsorted(cumulative, t) + 1
    axes[1].axhline(t, color='gray', linestyle='--', linewidth=0.8)
    axes[1].annotate(f'{t*100:.0f}% → {n_feat} features', xy=(n_feat, t),
                     xytext=(n_feat + 200, t - 0.03), fontsize=8,
                     arrowprops=dict(arrowstyle='->', color='gray'))
axes[1].set_xlabel('Nombre de features', fontsize=11)
axes[1].set_ylabel('Importance cumulée', fontsize=11)
axes[1].set_title('Importance Cumulée des Features', fontsize=12)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Sélection d'un Sous-ensemble de Features

On teste **3 seuils** de sélection :
- **Top-500** features (5% du vocabulaire)
- **Top-1000** features (10% du vocabulaire)
- **Top-2000** features (20% du vocabulaire)

On compare les performances de chaque sous-ensemble via Logistic Regression.

In [ ]:
def build_reduced_matrices(vectorizer, rf_model, X_train_raw, X_val_raw, X_test_raw, k):
    """Construit les matrices TF-IDF réduites aux top-k features selon RF importance."""
    all_features  = vectorizer.get_feature_names_out()
    importances   = rf_model.feature_importances_
    top_k_indices = np.argsort(importances)[::-1][:k]
    
    # Réindexer depuis la matrice TF-IDF déjà transformée
    X_tr = vectorizer.transform(X_train_raw)[:, top_k_indices]
    X_vl = vectorizer.transform(X_val_raw)[:, top_k_indices]
    X_ts = vectorizer.transform(X_test_raw)[:, top_k_indices]
    
    selected_names = all_features[top_k_indices]
    return X_tr, X_vl, X_ts, selected_names


K_VALUES = [500, 1000, 2000]
results = {}

for k in K_VALUES:
    X_tr_k, X_vl_k, X_ts_k, sel_names = build_reduced_matrices(
        vectorizer_full, rf, X_train, X_val, X_test, k
    )
    
    clf_k = LogisticRegression(max_iter=500, random_state=42, n_jobs=-1)
    clf_k.fit(X_tr_k, y_train)
    
    y_pred_k = clf_k.predict(X_vl_k)
    acc_k = accuracy_score(y_val, y_pred_k)
    f1_k  = f1_score(y_val, y_pred_k, average='macro')
    
    results[f'Top-{k}'] = {
        'n_features': k,
        'accuracy':   acc_k,
        'f1_macro':   f1_k,
        'clf':        clf_k,
        'X_val':      X_vl_k,
        'X_test':     X_ts_k,
        'selected_names': sel_names
    }
    print(f'Top-{k:5d} features | Acc: {acc_k:.4f} | F1 Macro: {f1_k:.4f}')

# Ajouter le baseline
results['Baseline (10k)'] = {
    'n_features': N_FEATURES_FULL,
    'accuracy':   acc_baseline,
    'f1_macro':   f1_baseline
}

## 7. Comparaison Avant / Après Sélection

In [ ]:
# Tableau comparatif
summary = pd.DataFrame([
    {
        'Configuration':   name,
        'Nb features':     v['n_features'],
        'Accuracy (val)':  round(v['accuracy'], 4),
        'F1 Macro (val)':  round(v['f1_macro'], 4),
        'Réduction (%)':   round((1 - v['n_features'] / N_FEATURES_FULL) * 100, 1)
    }
    for name, v in results.items()
])

print('=== COMPARAISON AVANT / APRÈS SÉLECTION ===')
display(summary.sort_values('F1 Macro (val)', ascending=False).reset_index(drop=True))

In [ ]:
# Visualisation
configs = list(results.keys())
f1_scores = [results[c]['f1_macro'] for c in configs]
acc_scores = [results[c]['accuracy'] for c in configs]
nb_feats   = [results[c]['n_features'] for c in configs]

x = np.arange(len(configs))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 scores
bars1 = axes[0].bar(x - width/2, acc_scores, width, label='Accuracy', color='steelblue', alpha=0.85)
bars2 = axes[0].bar(x + width/2, f1_scores,  width, label='F1 Macro', color='darkorange', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(configs, rotation=15, ha='right')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_title('Performance par Nombre de Features', fontsize=12)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

# F1 vs Nb features
axes[1].plot(nb_feats, f1_scores, marker='o', color='darkorange', linewidth=2, markersize=8, label='F1 Macro')
axes[1].plot(nb_feats, acc_scores, marker='s', color='steelblue', linewidth=2, markersize=8, label='Accuracy')
for i, c in enumerate(configs):
    axes[1].annotate(c, (nb_feats[i], f1_scores[i]), textcoords='offset points', xytext=(5, 5), fontsize=8)
axes[1].set_xlabel('Nombre de features', fontsize=11)
axes[1].set_ylabel('Score', fontsize=11)
axes[1].set_title('Trade-off Complexité vs Performance', fontsize=12)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Meilleur Sous-ensemble & Rapport Final

On sélectionne la configuration avec le **meilleur F1 Macro** parmi les versions réduites.

In [ ]:
# Identifier la meilleure configuration réduite
reduced_configs = {k: v for k, v in results.items() if k != 'Baseline (10k)'}
best_config_name = max(reduced_configs, key=lambda k: reduced_configs[k]['f1_macro'])
best_config = reduced_configs[best_config_name]

print(f'=== MEILLEURE CONFIGURATION RÉDUITE : {best_config_name} ===')
print(f'Nb features   : {best_config["n_features"]}')
print(f'Accuracy (val): {best_config["accuracy"]:.4f}')
print(f'F1 Macro (val): {best_config["f1_macro"]:.4f}')
print(f'Réduction     : {(1 - best_config["n_features"] / N_FEATURES_FULL) * 100:.1f}%')
print()

# Comparaison explicite avec le baseline
delta_f1  = best_config['f1_macro']  - f1_baseline
delta_acc = best_config['accuracy'] - acc_baseline
print(f'Δ F1 Macro vs baseline : {delta_f1:+.4f}')
print(f'Δ Accuracy vs baseline : {delta_acc:+.4f}')

In [ ]:
# Rapport de classification sur le jeu de validation
best_clf = best_config['clf']
y_pred_best_val = best_clf.predict(best_config['X_val'])

print(f'=== RAPPORT DE CLASSIFICATION — {best_config_name} (Validation) ===')
print(classification_report(y_val, y_pred_best_val))

In [ ]:
# Évaluation finale sur le test set
y_pred_best_test = best_clf.predict(best_config['X_test'])
f1_test  = f1_score(y_test, y_pred_best_test, average='macro')
acc_test = accuracy_score(y_test, y_pred_best_test)

print(f'=== PERFORMANCE FINALE SUR TEST ({best_config_name}) ===')
print(f'Accuracy (test) : {acc_test:.4f}')
print(f'F1 Macro (test) : {f1_test:.4f}')

In [ ]:
# Matrice de confusion
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_best_test)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Matrice de Confusion — {best_config_name} (Test)')
plt.ylabel('Vrai')
plt.xlabel('Prédit')
plt.tight_layout()
plt.show()

## 9. Top Features Sélectionnées

In [ ]:
selected_names = best_config['selected_names']
print(f'Features sélectionnées ({best_config_name}) — échantillon :')
print(', '.join(selected_names[:30]))

## 10. Sauvegarde

In [ ]:
# Sauvegarder le meilleur modèle réduit et la liste des features sélectionnées
model_path = os.path.join(MODELS_DIR, 'best_selected_classifier.pkl')
feat_path  = os.path.join(MODELS_DIR, 'selected_feature_names.pkl')
vec_path   = os.path.join(MODELS_DIR, 'tfidf_vectorizer_full.pkl')

joblib.dump(best_clf,        model_path)
joblib.dump(selected_names,  feat_path)
joblib.dump(vectorizer_full, vec_path)

print(f'Modèle sauvegardé         : {model_path}')
print(f'Features sélectionnées    : {feat_path}')
print(f'Vectorizer sauvegardé     : {vec_path}')

---
## ✅ Résumé

| Étape | Résultat |
|-------|----------|
| Feature importance calculée | Random Forest sur TF-IDF (10k features) |
| Top-N features visualisées | Top 30 features affichées |
| Configurations testées | Top-500, Top-1000, Top-2000 |
| Meilleure config. réduite | Voir cellule 8 |
| Comparaison avant/après | Tableau + graphique section 7 |
| Modèle sauvegardé | `models/best_selected_classifier.pkl` |